<a href="https://colab.research.google.com/github/YYUU1407/1_chess_midterm/blob/main/infomtalc2026_midterm_chess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFOMTALC 2026: Midterm assignment 1
## Transformer-based chess player

## 1. General set-up
This Colab contains a description of one of the exercises for your midterm assignment. In this exercise, you will need to write a transformer-based chess player!  

That is, you will need to implement a class called `TransformerPlayer` -- a chess player that will look at a chess board and produce a move. Players submitted by all students will play in a championship against each other and against some baseline players that we will inject into the championship. The championship will result in a leaderboard, and points will be assigned based on this leaderboard and validation checks.

The idea behind this assignment is not chess: at least one of the authors of this assignment can barely remember how figures move. This is an artificial case of an in-domain problem that has to be approached with a particular toolbox -- in our case, transformer-based models. In fact, chess is a great example of a task that can be seen as a text task: both the state of the board and the move can be represented as text sequences (we will talk about the format below).

You are free to choose the architecture and the approach yourself. For instance, maybe you want an encoder model to classify board states into as many classes as there are available moves? Maybe you want a decoder model to continue the board-state sequence with what the next move should be? Or an encoder-decoder model to translate board states into moves? It's up to you.

Whether you want to do some training or make creative use of already existing transformer models is up to you as well. **It is not allowed** to use already trained specialized chess transformer models (there are some on HF). To check for that, we will require you to submit a link to the model you are using on HF. So, you will be **required** to push your model to HF if you are doing some additional training.

If you're doing training, again, you take care of where to get the data. There are existing datasets, or you can decide to generate your dataset yourself based on some helping code we provide below.

Ok, enough of general description, let's move on to something more concrete. We pre-wrote some code to orchestrate this assignment and to help you. Clone the repository and install everything:


In [ ]:
!git clone https://github.com/bylinina/chess_exam.git
%cd /content/chess_exam
!pip install -e .

Cloning into 'chess_exam'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 53 (delta 15), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 24.80 KiB | 130.00 KiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/chess_exam
Obtaining file:///content/chess_exam
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 110.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of smolagents to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.1/533.1 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 12.5 MB/s eta 0:00:00
  Created wheel for chess: filename=chess-1.11.2-py3-no

In [ ]:
from chess_tournament import (
    Game,
    Player,
    RandomPlayer,
    LMPlayer,
    SmolPlayer,
    EnginePlayer,
    run_tournament
    )

To make things more structured, we made a `Player` class that all players inherit from. All it has is a built-in `.get_move()` method:

```python
class Player(ABC):
    def __init__(self, name: str):
        self.name = name

    @abstractmethod
    def get_move(self, fen: str) -> Optional[str]:
        pass
```

Now, one of our baseline players will be the RandomPlayer. It is defined based on Player but now the mechanics of producing a move given an input position is defined as well (it just produces a random move from the set of valid moves given a position):

```python
class RandomPlayer(Player):
    def get_move(self, fen: str) -> Optional[str]:
        board = chess.Board(fen)
        moves = list(board.legal_moves)
        return random.choice(moves).uci() if moves else None
```

As you see, it takes a string as input and produces either a random legal move or None if there are no legal moves available.

Time to talk about **the input sequence** -- board state. We are using the standard **FEN notation**. It's a line of text that fully describes the state of the board. Read more about it [here](https://www.chess.com/terms/fen-chess). For example, this:

```8/8/8/4p1K1/2k1P3/8/8/8 b - - 0 1```

corresponds to this:

<img src="https://images.chesscomfiles.com/uploads/v1/images_users/tiny_mce/pdrpnht/phpdAnlDc.png" width="200"/>

The moves have [UCI format](https://en.wikipedia.org/wiki/Universal_Chess_Interface), for instance:

- e2e4
- e7e5

Now, in order for the players to actually play, we need to initialize them. Let's make two instances of random player, to see how they will play with each other. We have to give each of them a name:

In [ ]:
random1 = RandomPlayer("Random-1")
random2 = RandomPlayer("Random-2")

In order to execute the game, we give you the `Game` class. It orchestrates the exchange of board states and moves between the players. Most importantly, it has the `play` method that runs the game! Here is how it works: you give it two player instances, along with an optional `max_half_moves` parameter so that it doesn't go on for too long (there's a default, just in case).

Then you play the game! There are behavioral controls there as well(`verbose` parameter to tell it whether to print out intermediate steps; `log_moves` to print a short log per move -- useful if you want to generate training data!!; `log_to_file` to pass the log file name if you want it; feel free to explore the code in the repo as well):

In [ ]:
game = Game(random1, random2, max_half_moves=50)
game.play()

('1/2-1/2', {'Random-1': 0.5, 'Random-2': 0.5}, {'Random-2': 0, 'Random-1': 0})

In [ ]:
game.play(log_moves=True, log_to_file='log.csv')

PLY 000 | Random-1 | white | rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1 | h2h3 | fallback:False
PLY 001 | Random-2 | black | rnbqkbnr/pppppppp/8/8/8/7P/PPPPPPP1/RNBQKBNR b KQkq - 0 1 | f7f6 | fallback:False
PLY 002 | Random-1 | white | rnbqkbnr/ppppp1pp/5p2/8/8/7P/PPPPPPP1/RNBQKBNR w KQkq - 0 2 | c2c4 | fallback:False
PLY 003 | Random-2 | black | rnbqkbnr/ppppp1pp/5p2/8/2P5/7P/PP1PPPP1/RNBQKBNR b KQkq - 0 2 | e7e5 | fallback:False
PLY 004 | Random-1 | white | rnbqkbnr/pppp2pp/5p2/4p3/2P5/7P/PP1PPPP1/RNBQKBNR w KQkq - 0 3 | d2d4 | fallback:False
PLY 005 | Random-2 | black | rnbqkbnr/pppp2pp/5p2/4p3/2PP4/7P/PP2PPP1/RNBQKBNR b KQkq - 0 3 | g7g5 | fallback:False
PLY 006 | Random-1 | white | rnbqkbnr/pppp3p/5p2/4p1p1/2PP4/7P/PP2PPP1/RNBQKBNR w KQkq - 0 4 | h1h2 | fallback:False
PLY 007 | Random-2 | black | rnbqkbnr/pppp3p/5p2/4p1p1/2PP4/7P/PP2PPPR/RNBQKBN1 b Qkq - 1 4 | e5d4 | fallback:False
PLY 008 | Random-1 | white | rnbqkbnr/pppp3p/5p2/6p1/2Pp4/7P/PP2PPPR/RNBQKBN1 w Qkq - 

('0-1', {'Random-1': 0.0, 'Random-2': 1.0}, {'Random-1': 0, 'Random-2': 0})

You might notice the `fallback:Bool` part of the log. It's there to keep track of the times the player either outputs an illegal move or doesn't produce a move (outputs `None`) and the list of available moves is not empty. In this case, `Game` generously falls back to a mechanism that picks a random legal move instead of recording that the player simply lost.

Note that our `Random` players are actually pretty strong -- at least they always produce a legal move! So it's a solid baseline in a situation when fallback counts are taken into account for ranking (and we **will** take them into account -- otherwise a solid strategy for your player would be to simply always output `None` to induce a fallback).

Feel free to use our `Random` player to test your player against! But we have more players for you to play with.

## 2. Pre-made baseline players

We have pre-made some player classes you can practice against:
- `RandomPlayer` (you saw it above)
- `EnginePlayer`
- `LMPlayer`
- `SmolPlayer`

### EnginePlayer

`EnginePlayer` uses a very strong chess player engine available via an API. You can't base your submission on it -- but we will inject two versions of this player into our championship. It runs with a free API so it sometimes limits the number of calls and can produce fallbacks. Even with that, it's **very strong**! We make it weaker by manipulating the probability it produces its second-best move and a random move.

If you want to use it, you can get a free API key for StockFish16 chess engine via [RapidAPI](https://rapidapi.com/cinnamon17/api/Chess%20StockFish%2016%20API) (to see your API key press the blue 'Test endpoint' button). Then input your API key here and you're ready to go:

In [ ]:
from getpass import getpass
import os

os.environ["RAPIDAPI_KEY"] = getpass("RapidAPI key:")

RapidAPI key:··········


In [ ]:
engine_strong = EnginePlayer("Stockfish-Strong", blunder_rate=0.0, ponder_rate=0.05)
engine_weak = EnginePlayer("Stockfish-Weak", blunder_rate=0.25, ponder_rate=0.65)

game = Game(engine_strong, random1, max_half_moves=200) # 200 is a more useful number of moves than 50 we had before, but will take longer
game.play() # --> (string of the outcome, points gained per player, fallback count)

('0-1',
 {'Stockfish-Strong': 1.0, 'Random-1': 0.0},
 {'Random-1': 0, 'Stockfish-Strong': 1})

### LMPlayer

Here is a super-quick baseline LM player: just a quantized big-ish LM -- `mistralai/Mistral-7B-Instruct-v0.2` -- with a couple of tricks to reduce the rate of illegal moves produced (try `n` times before outputting `None`, some prompt-play). Don't use this class for your submissions! Your class should inherit directly from `Player`, not from `LMPlayer`. This one doesn't load the model very efficiently, so it's just for you to play against if you want.

In [ ]:
lm = LMPlayer("Mistral-7B")

[Mistral-7B] Loading mistralai/Mistral-7B-Instruct-v0.2 on cuda
[Mistral-7B] Quantization mode: 4bit


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
Game(engine_strong, lm).play()

('0-1',
 {'Stockfish-Strong': 1.0, 'Mistral-7B': 0.0},
 {'Mistral-7B': 26, 'Stockfish-Strong': 1})

### SmolPlayer

This baseline player is using a waaaay larger model (`moonshotai/Kimi-K2-Instruct`) that's accessed via Inference API, but without any bells and whistles and postprocessing. In order to use this player, you need a HF token, and small costs might be incurred. If you can't use this player, no worries -- you don't really need it, it's just for illustration and for you to try your player against. It will, however, enter the championship as a baseline!

**You can't use this kind of player in your submission**: Your models have to be running on a free-tier Colab locally! Otherwise, it'd be a competition of API costs you are willing to allocate -- huge models are probably pretty good out of the box (maybe you can use them for your data generation though? up to you)

In [ ]:
os.environ["HF_TOKEN"] = getpass("HF token:")

HF token:··········


In [ ]:
kimi = SmolPlayer("Kimi")
Game(kimi, lm).play()

('0-1', {'Kimi': 1.0, 'Mistral-7B': 0.0}, {'Mistral-7B': 85, 'Kimi': 40})

## 3. Submission and grading

### 3.1 Submit your player!

Your submission for this exercise will happen via **[Google Forms here](https://forms.gle/Cg425deqMeYF5ujRA)**. You will need to fill in:
- your first and last name,
- student number
- the link to the GitHub repository with your player class (in this format: `https://github.com/<USERNAME>/<REPO_NAME>.git`)
- the link to the HF model card for the transformer model that is the basis of your player
- (+ some additional info about your solution)

It is very important that:
- Your repository contains a file with the name `player.py`
- In that file, you define a class that has the name `TransformerPlayer`
- That class inherits from the class `Player`
- That class contains the `get_move()` method
- Your repository contains the `requirements.txt` file with all packages that need to be installed in a standard Colab setting
- Your player has to be initializable with just the name as the only argument (`player = TransformerPlayer("Student")` or something like that -- so the rest of the arguments have to be set by default)

If your `TransformerPlayer` doesn't pass these tests (they will be automatic), your player will be disqualified. In order to help you, we created a template repo that passess all the tests. You can just fork it or clone it or just copy the code and work with it and then use the result for your submission: https://github.com/bylinina/chess_player_template.git

### 3.2 Get your rating and grade

- If your player is disqualified, you get 0 points
- If your player participates in the championship, your grade will be defined by the distribution of the players in the leaderboard with respect to each other and with respect to the injected baselines
- If you want to know the details of how the validation and championships will be done, check out [this Colab with a fake trial run](https://colab.research.google.com/drive/1IeLAYsjcWuorTd7JXV8TNfxQvmzewzfe?usp=sharing). We are most likely to run it again, as is.
- We will share the leaderboard and results as soon as we have them!

### 3.3 Tips, dos and don'ts
- Our transormer-based example players are all built on decoder-only models. Not because they are necessarily better! But just because it was the fastest things to assemble. Don't feel the need to only work with decoders -- as long as you can make the model output a string, it's fine -- it can be an encoder class label
- Just to reiterate: Don't use pre-trained **chess transformer** models! We will check
- Don't make 'dummy players' that always ouput `None` and therefore default to the `RandomPlayer` strong baseline. We will check
- You don't have to train / fine-tune a model yourself. You can find ways to strengthen a general LM by all sorts of tricks: prompting, comparing probabilities of competitor output strings (check out the [`minicons`](https://github.com/kanishkamisra/minicons) library that might be useful), enforcing the output format (check out the [`lm-format-enforcer`](https://github.com/noamgat/lm-format-enforcer) library) -- anything!
- You might as well want to fine-tune! Models that you can fine-tune in a Colab would be smaller than the ones that can fit the Colab for inference, but maybe specialized knowledge beats size? We don't know! If you want to train, you are free to search data anywhere you want: there are existing datasets, but also you can use the existing player baselines as a means of generating board+move data.
- If you fine-tune a model, you have to make it public on HF, otherwise the championship won't work. Push it it HF.


Ok, that's all! Best of luck!! (Maybe we'll participate too, who knows)

In [ ]:
from __future__ import annotations

import os
import re
import random
from typing import Optional, List

import chess

from transformers import AutoTokenizer, AutoModelForCausalLM


_UCI_RE = re.compile(r"\b([a-h][1-8][a-h][1-8][qrbn]?)\b", re.IGNORECASE)


class TransformerPlayer(Player):
    """
    Transformer-based chess player.
    - Input: FEN string
    - Output: UCI move string (e.g., 'e2e4', 'e7e8q') or None if no legal moves
    """

    def __init__(
        self,
        name: str,
        model_name: str = "Qwen/Qwen3.5-4B",
        seed: int = 42,
        use_4bit: bool = True,
        max_new_tokens: int = 8,
        temperature: float = 0.2,
        top_p: float = 0.9,
    ):
        super().__init__(name)

        random.seed(seed)
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = top_p

        # Device + model loading (4-bit when possible to fit in Colab)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

        load_kwargs = {"device_map": "auto"}
        if use_4bit:
            # bitsandbytes must be installed; if not, it will fall back below
            load_kwargs.update({"load_in_4bit": True})

        try:
            self.model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
        except Exception:
            # Fallback: load without 4-bit (may be slower / may not fit on GPU for large models)
            self.model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

        self.model.eval()

    def _legal_moves_uci(self, board: chess.Board) -> List[str]:
        return [m.uci() for m in board.legal_moves]

    def _build_prompt(self, fen: str, legal_moves: List[str]) -> str:
        # Keep prompt short-ish (speed) but strict (format)
        return (
            "You are a chess engine.\n"
            "Given this chess position in FEN:\n"
            f"{fen}\n\n"
            "Choose exactly ONE move from the following LEGAL moves (UCI):\n"
            f"{', '.join(legal_moves)}\n\n"
            "Return ONLY the move in UCI format. No explanation."
        )

    def _extract_uci(self, text: str) -> Optional[str]:
        text = text.strip().lower()
        m = _UCI_RE.search(text)
        return m.group(1).lower() if m else None

    def get_move(self, fen: str) -> Optional[str]:
        board = chess.Board(fen)
        legal = self._legal_moves_uci(board)
        if not legal:
            return None

        prompt = self._build_prompt(fen, legal)

        inputs = self.tokenizer(prompt, return_tensors="pt")
        # Put tensors on the same device as the model
        if hasattr(self.model, "device"):
            inputs = {k: v.to(self.model.device) for k, v in inputs.items()}

        try:
            out = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=True,
                temperature=self.temperature,
                top_p=self.top_p,
                pad_token_id=self.tokenizer.eos_token_id,
            )
            text = self.tokenizer.decode(out[0], skip_special_tokens=True)
            move = self._extract_uci(text)

            # Strict legality check (prevents illegal moves)
            if move in legal:
                return move

        except Exception:
            # If model generation fails for any reason, fall back safely
            pass

        # Safe fallback to guarantee a legal move
        return random.choice(legal)